In [ ]:
!pip uninstall -y torch torchvision bitsandbytes transformers peft trl accelerate datasets huggingface_hub tokenizers wandb xformers
!pip install --force-reinstall --no-cache-dir \
    torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install --force-reinstall --no-cache-dir \
    bitsandbytes==0.45.0 transformers==4.46.3 peft==0.13.2 \
    trl==0.12.2 accelerate==1.1.1 datasets==3.1.0 \
    huggingface_hub==0.27.1 tokenizers==0.20.3 wandb==0.18.7

In [3]:
"""
Merge QLoRA adapter into base Mistral-7B-Instruct-v0.3 fp16, push merged to HF.
Run on Kaggle T4x2, ~20-25 min.
"""
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER = "ayushgupta7777/sentinelops-mistral7b-qlora-adapter"
MERGED_REPO = "ayushgupta7777/sentinelops-mistral7b-merged"
LOCAL = "/kaggle/working/merged"

print("Loading base in fp16...")
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)

print("Attaching adapter + merging...")
peft_model = PeftModel.from_pretrained(base, ADAPTER)
merged = peft_model.merge_and_unload()

print("Saving merged locally (14 GB)...")
merged.save_pretrained(LOCAL, safe_serialization=True, max_shard_size="4GB")
tok.save_pretrained(LOCAL)

# Free VRAM before upload
del base, peft_model, merged
torch.cuda.empty_cache()

print("Uploading to HF...")
api = HfApi()
api.create_repo(MERGED_REPO, exist_ok=True, private=False)
api.upload_folder(folder_path=LOCAL, repo_id=MERGED_REPO, repo_type="model")
print(f"Done: https://huggingface.co/{MERGED_REPO}")

2026-04-19 21:11:10.359085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776633070.560195     468 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776633070.615718     468 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776633071.082428     468 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776633071.082461     468 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776633071.082464     468 computation_placer.cc:177] computation placer alr

Loading base in fp16...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.93G [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Done: https://huggingface.co/ayushgupta7777/sentinelops-mistral7b-merged


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)

In [ ]:
!pip show torch | head -5
!pip index versions torch --index-url https://download.pytorch.org/whl/cu124 2>&1 | head -3

In [ ]:
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -5
!pip install --no-cache-dir --ignore-installed \
    --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.5.1 2>&1 | tail -10

In [ ]:
!pip show transformers peft bitsandbytes accelerate 2>&1 | grep -E "Name|Version"

In [2]:
import torch
print(torch.__version__, torch.version.cuda)

2.5.1+cu124 12.4
